# PSGC API → Landing

Extract the official PSA API into landing files. Run all cells in order.
No Spark, Bronze table, or Delta metadata table is required.

Set `config_path` widget or `PSGC_CONFIG_PATH` to your `sources.yaml`.
Required settings under top-level `psgc` or `sources.psgc`: `base_url`, `version`, `endpoint`,
`landing_path`. Optional: `source_name`, `page_size`, and `max_pages`.
Use an existing writable `/Volumes/<catalog>/<schema>/<volume>/...` path in
Databricks, or a local filesystem path outside Databricks. `dbfs:/` and cloud
URIs are not supported by the Python file writer.

Supply your own valid PSA token securely. The notebook first checks the runtime
`PSA_PSGC_TOKEN` environment variable. On Databricks serverless, use the optional
`secret_scope` and `secret_key` widgets to identify an existing readable workspace
secret holding your token. No scope/key names are hardcoded or required in YAML.
The actual token must never go in widgets, YAML, notebook source or job parameters.
Serverless does not support compute-scoped environment variables.
Missing/unreadable credentials stop execution before landing writes. Each workspace
supplies its own credentials and landing access. The configured `Q2_2024` version
is a fixed version, not an automatic latest-version selection.

Each attempt gets a unique `raw/version=.../run_id=...` folder:
- `page_0001.json`: API response with echoed credentials redacted.
- `records_page_0001.ndjson`: one source record per line, without field renaming.
- `manifest.json`: batch status, counts, source version, checksum and file list.
- `_SUCCESS`: created only after all pages and landed counts reconcile.

Incomplete/failed batches are retained without `_SUCCESS`; downstream readers must
require both `_SUCCESS` and a `landed` manifest. Reruns retain separate snapshots.
Business validation and Bronze loading can follow as separate notebooks.
Do not enable HTTP debug logging because requests carry the token in the query.


In [0]:
dbutils.widgets.text(
    "config_path",
    "/Workspace/Users/shiena.arico@ftwfoundation.org/philippine-public-health-platform/config/sources_geography.yaml"
)

dbutils.widgets.text("secret_scope", "g5-public-health")
dbutils.widgets.text("secret_key", "psa-psgc-token")

In [0]:
import os
import json
import hashlib
import uuid
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote, quote_plus, urlsplit

import requests
import yaml
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


In [0]:
# ============================================================
# CELL 2 — REUSABLE HELPER FUNCTIONS
# ============================================================


# ------------------------------------------------------------
# HTTP SESSION WITH RETRIES
# ------------------------------------------------------------

retry_strategy = Retry(
    total=3,
    connect=3,
    read=3,
    status=3,
    backoff_factor=2,
    status_forcelist=[
        429,
        500,
        502,
        503,
        504,
    ],
    allowed_methods=["GET"],
    raise_on_status=False,
)

session = requests.Session()

session.mount(
    "https://",
    HTTPAdapter(
        max_retries=retry_strategy
    ),
)


# ------------------------------------------------------------
# FETCH ONE PSGC API PAGE
# ------------------------------------------------------------

def fetch_psgc_page(
    api_url,
    token,
    page,
    page_size,
):
    """
    Fetch one page from the PSA PSGC API.

    Returns:
        payload   - parsed JSON response
        raw_text  - response body with echoed credentials redacted

    The API token is never printed or returned in logs.
    """

    params = {
        "token": token,
        "page": page,
        "page_size": page_size,
    }

    if not isinstance(token, str) or not token.strip():
        raise RuntimeError("PSGC API request blocked: missing token.")
    try:
        response = session.get(
            api_url, params=params, timeout=60, allow_redirects=False,
        )
    except requests.exceptions.RequestException:
        # Requests errors can include the URL with its token query parameter.
        raise RuntimeError("PSA API transport failed; check connectivity.") from None

    status_code = response.status_code

    content_type = response.headers.get(
        "Content-Type",
        "",
    )


    # Do not print response.url because
    # the token may be present in the query string.
    if status_code != 200:
        raise RuntimeError(
            f"PSA API request failed "
            f"with HTTP {status_code}."
        )


    if "application/json" not in content_type.lower():
        raise RuntimeError(
            f"Expected JSON response but received "
            "a non-JSON content type."
        )


    try:
        payload = response.json()

    except ValueError:
        raise RuntimeError(
            "PSA response could not be parsed as JSON."
        ) from None


    # API pagination URLs may echo credentials. Redact before persisting.
    raw_text = response.text
    for sensitive in sorted({token, quote(token, safe=""), quote_plus(token)}, key=len, reverse=True):
        raw_text = raw_text.replace(sensitive, "[REDACTED]")
    try:
        payload = json.loads(raw_text)
    except ValueError:
        raise RuntimeError("PSA response could not be safely redacted.") from None
    return payload, raw_text


# ------------------------------------------------------------
# EXTRACT PSGC RECORDS FROM API RESPONSE
# ------------------------------------------------------------

def extract_psgc_records(payload):
    """
    Extract PSGC records from the PSA paginated response.

    Actual response structure observed:

    {
        "count": ...,
        "next": ...,
        "previous": ...,
        "results": [...]
    }
    """

    if not isinstance(payload, dict):
        raise TypeError(
            "Unexpected PSA response type: "
            f"{type(payload)}"
        )


    results = payload.get(
        "results"
    )


    # Actual response structure observed
    if isinstance(results, list):
        return results


    # Defensive fallback in case another API version
    # returns a nested results object.
    if isinstance(results, dict):

        psgc_data = results.get(
            "psgc_data"
        )

        if isinstance(
            psgc_data,
            list,
        ):
            return psgc_data


        nested_data = results.get(
            "data"
        )

        if isinstance(
            nested_data,
            list,
        ):
            return nested_data


    raise ValueError(
        "Could not locate PSGC records. "
        f"Top-level keys: {list(payload.keys())}"
    )




In [0]:
def widget_value(name, default=""):
    utils = globals().get("dbutils")
    if utils is None:
        return default
    try:
        return utils.widgets.get(name).strip()
    except Exception:
        utils.widgets.text(name, default, name.replace("_", " ").title())
        return utils.widgets.get(name).strip()


CONFIG_PATH = (
    widget_value("config_path")
    or os.environ.get("PSGC_CONFIG_PATH", "").strip()
    or "config/sources.yaml"
)
if not os.path.isfile(CONFIG_PATH):
    raise ValueError(
        "Config file not found. Set config_path or PSGC_CONFIG_PATH "
        "to your sources.yaml."
    )

with open(
    CONFIG_PATH,
    "r",
    encoding="utf-8",
) as file:
    config = yaml.safe_load(file)


if not isinstance(config, dict):
    raise ValueError("sources.yaml must contain a YAML mapping.")
source_config = config.get("sources", config)
if not isinstance(source_config, dict) or not isinstance(source_config.get("psgc"), dict):
    raise ValueError("Expected psgc settings at the top level or under sources.psgc.")
psgc_cfg = source_config["psgc"]

SOURCE_NAME = str(psgc_cfg.get("source_name", "psa_psgc"))
BASE_URL = str(psgc_cfg["base_url"]).rstrip("/")
SOURCE_VERSION = str(psgc_cfg["version"]).strip()
ENDPOINT = str(psgc_cfg["endpoint"]).strip("/")
PAGE_SIZE = int(psgc_cfg.get("page_size", 1000))
MAX_PAGES = int(psgc_cfg.get("max_pages", 10000))
LANDING_PATH = str(psgc_cfg["landing_path"]).strip()
if PAGE_SIZE < 1 or MAX_PAGES < 1:
    raise ValueError("page_size and max_pages must be positive.")
if not SOURCE_VERSION or any(c not in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-" for c in SOURCE_VERSION):
    raise ValueError("version must contain only letters, numbers, underscores or hyphens.")
if not ENDPOINT or "?" in ENDPOINT or "#" in ENDPOINT:
    raise ValueError("endpoint must be a path without query parameters or fragments.")
url_parts = urlsplit(BASE_URL)
if url_parts.scheme != "https" or not url_parts.hostname or url_parts.query or url_parts.fragment or url_parts.username:
    raise ValueError("base_url must be an HTTPS URL without credentials, query or fragment.")
if not LANDING_PATH or ":" in LANDING_PATH:
    raise ValueError("landing_path must be a local or /Volumes/ path, not a URI.")
API_URL = f"{BASE_URL}/{SOURCE_VERSION}/{ENDPOINT}"
print("Landing configuration loaded.")


In [0]:
# Token from runtime environment, or configurable workspace Secrets.
# These widgets contain secret identifiers only, never the actual token.
PSA_TOKEN = None
PSA_TOKEN = os.environ.get("PSA_PSGC_TOKEN", "").strip()
if not PSA_TOKEN:
    secret_scope = widget_value("secret_scope")
    secret_key = widget_value("secret_key")
    if secret_scope and secret_key and globals().get("dbutils") is not None:
        try:
            value = dbutils.secrets.get(scope=secret_scope, key=secret_key)
            PSA_TOKEN = value.strip() if isinstance(value, str) else ""
        except Exception:
            raise RuntimeError(
                "Cannot read configured PSA secret. Check secret_scope, secret_key "
                "and your workspace permissions. No landing writes started."
            ) from None
if not PSA_TOKEN:
    raise RuntimeError(
        "No PSA token available. On Databricks serverless, configure secret_scope "
        "and secret_key widgets for an existing readable secret holding your PSA token. "
        "Other runtimes may supply PSA_PSGC_TOKEN securely. No landing writes started."
    ) from None
print("PSA credential available; ready for API extraction.")


## Extract and verify landing

Page completeness and file counts are checked before success. Source fields are
retained as returned; missing codes, duplicates or version differences can be
handled later in Bronze validation. Failed requests retain partial files for review.


In [0]:
def extract_to_landing(fetch_page, api_url, token, landing_path,
                       source_name, source_version, endpoint,
                       page_size=1000, max_pages=10000):
    if not isinstance(token, str) or not token.strip():
        raise RuntimeError("Missing token: no landing writes started.")
    run_id = str(uuid.uuid4())
    batch_path = Path(landing_path) / "raw" / f"version={source_version}" / f"run_id={run_id}"
    batch_path.mkdir(parents=True, exist_ok=False)
    manifest = {
        "run_id": run_id, "source_name": source_name,
        "source_version": source_version, "endpoint": endpoint,
        "started_at": datetime.now(timezone.utc).isoformat(),
        "status": "in_progress", "page_count": 0, "record_count": 0,
        "expected_record_count": None, "files": [],
    }

    def save_manifest():
        pending = batch_path / "manifest.pending"
        pending.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
        os.replace(pending, batch_path / "manifest.json")

    save_manifest()
    record_hashes = []
    page_signatures = set()
    try:
        for page in range(1, max_pages + 1):
            payload, raw_text = fetch_page(api_url, token, page, page_size)
            if not isinstance(payload, dict) or "next" not in payload or "count" not in payload:
                raise ValueError("Expected paginated API response with count and next.")
            count = payload["count"]
            if isinstance(count, bool) or not isinstance(count, int) or count < 0:
                raise ValueError("API count must be a nonnegative integer.")
            if page == 1:
                manifest["expected_record_count"] = count
            elif count != manifest["expected_record_count"]:
                raise RuntimeError("API count changed during pagination; rerun for a consistent batch.")
            records = extract_psgc_records(payload)
            if any(not isinstance(record, dict) for record in records):
                raise ValueError("API records must be JSON objects.")
            canonical = [json.dumps(record, sort_keys=True, separators=(",", ":"),
                                    ensure_ascii=False) for record in records]
            signature = hashlib.sha256("\n".join(canonical).encode("utf-8")).hexdigest()
            if records and signature in page_signatures:
                raise RuntimeError("Repeated API page detected; batch is incomplete.")
            page_signatures.add(signature)
            raw_name = f"page_{page:04d}.json"
            records_name = f"records_page_{page:04d}.ndjson"
            (batch_path / raw_name).write_text(raw_text, encoding="utf-8")
            with (batch_path / records_name).open("w", encoding="utf-8") as output:
                for record_text in canonical:
                    output.write(record_text + "\n")
                    record_hashes.append(hashlib.sha256(record_text.encode("utf-8")).hexdigest())
            manifest["files"].extend([raw_name, records_name])
            manifest["page_count"] += 1
            manifest["record_count"] += len(records)
            save_manifest()
            print(f"Page {page}: {len(records)} records saved.")
            if payload["next"] is None:
                break
            if not records:
                raise RuntimeError("Empty page with more pages advertised; batch is incomplete.")
        else:
            raise RuntimeError("Pagination limit reached before the final page.")

        if manifest["record_count"] != manifest["expected_record_count"]:
            raise RuntimeError("API total and extracted record count do not match.")
        landed_count = 0
        for name in manifest["files"]:
            path = batch_path / name
            if not path.is_file():
                raise RuntimeError("A landed page file is missing.")
            if name.endswith(".ndjson"):
                with path.open(encoding="utf-8") as landed:
                    for line in landed:
                        json.loads(line)
                        landed_count += 1
        if landed_count != manifest["record_count"]:
            raise RuntimeError("Landed record count does not match extraction.")
        manifest.update(
            status="landed", landed_record_count=landed_count,
            content_checksum=hashlib.sha256("\n".join(sorted(record_hashes)).encode("utf-8")).hexdigest(),
            finished_at=datetime.now(timezone.utc).isoformat(),
        )
        save_manifest()
        (batch_path / "_SUCCESS").write_text(run_id + "\n", encoding="utf-8")
        return str(batch_path), manifest
    except Exception:
        manifest.update(status="failed", finished_at=datetime.now(timezone.utc).isoformat())
        # Do not persist exception text: it may contain sensitive provider details.
        try:
            save_manifest()
        except OSError:
            pass
        raise


FINAL_RAW_PATH, LANDING_MANIFEST = extract_to_landing(
    fetch_psgc_page, API_URL, PSA_TOKEN, LANDING_PATH,
    SOURCE_NAME, SOURCE_VERSION, ENDPOINT, PAGE_SIZE, MAX_PAGES,
)
print("API → landing completed.")
print("Landing folder:", FINAL_RAW_PATH)
print("Pages:", LANDING_MANIFEST["page_count"])
print("Records:", LANDING_MANIFEST["record_count"])
print("Status:", LANDING_MANIFEST["status"])
